## Import Necessary Librires

In [9]:
import pandas as pd
import numpy as np
import warnings
import psutil
import time
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

warnings.filterwarnings('ignore')

%matplotlib inline

## Loading Data

In [2]:
df = pd.read_csv(r'../../3_Data/processed/g_2025_hourly_all_PCA_reduced.csv')
df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,taxi_demand
0,-0.518754,0.917431,-4.289254,1.130338,-0.421782,-1.711911,2.613329,-1.037458,-1.132565,1.374478,1051
1,-0.699536,0.915800,-4.206067,1.155734,-0.421137,-1.694405,2.545232,-1.031434,-1.098232,1.314072,436
2,-0.896397,0.911873,-4.015032,0.924829,-0.424154,-1.656173,2.435000,-1.014045,-1.031267,1.189971,268
3,-1.022439,0.904980,-3.811575,0.575039,-0.437192,-1.594720,2.337239,-0.991172,-0.964198,1.051892,220
4,-1.084073,0.895503,-3.635308,0.278344,-0.456338,-1.521287,2.255461,-0.967715,-0.905209,0.920323,277


In [3]:
df.shape

(6528, 11)

## **Splitig The Data**

In [4]:
X = df.drop(columns= ["taxi_demand",])
y = df['taxi_demand']

In [5]:
# Load and preprocess your data (X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
X_train.shape, X_test.shape

((5222, 10), (1306, 10))

In [7]:
y_train.shape, y_test.shape

((5222,), (1306,))

## Model Tuning

In [ ]:
def memory_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

start_time = time.time()
start_mem = memory_mb()

rf = RandomForestRegressor(random_state=42)

param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [10, 15, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_

# Evaluation
y_pred = best_rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Tuned RandomForest Performance")
print(f"RMSE: {rmse}")
print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")

print("Best Params:", search.best_params_)

print(f"Runtime: {time.time() - start_time:.2f}s")
print(f"Memory: {memory_mb() - start_mem:.2f} MB")


Fitting 5 folds for each of 40 candidates, totalling 200 fits
[CV] END bootstrap=False, max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time=  16.8s
[CV] END bootstrap=False, max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time=  17.2s
[CV] END bootstrap=False, max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time=  17.6s
[CV] END bootstrap=False, max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time=  18.4s
[CV] END bootstrap=False, max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time=  20.2s
[CV] END bootstrap=False, max_depth=30, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=500; total time=  23.7s
[CV] END bootstrap=False, max_depth=30, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=5

KeyboardInterrupt: 